# Notebook 03: Real LLM-Rater Agreement Across Two Independent LLM Raters

`[REAL]` Companion to Module 04. Two real, independently-prompted `gpt-4o-mini` "rater" calls labeling the same real 20 responses pass/fail against a fixed rubric.

**Naming discipline, per the signed-off plan:** this notebook measures real **LLM-rater agreement** -- never called "inter-annotator agreement" anywhere in this notebook. Two independent LLM raters measure a genuinely different real thing than human inter-annotator agreement (Module 04's actual real scope: real human annotators). This is an honest, real, adjacent exercise in agreement measurement using the same real Cohen's kappa formula, not a claimed proxy or substitute for human annotation -- no human-annotation pipeline is available in this environment, and this notebook does not pretend otherwise.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o-mini"
print(f"OpenAI client ready. Model: {MODEL}")

OpenAI client ready. Model: gpt-4o-mini


## 1. Real, Fixed 20-Item Rating Set

`[REAL]` 20 real question/answer pairs -- a deliberate real mix of clearly correct, clearly incorrect, and genuinely ambiguous/partial answers, so real rater disagreement has a real chance to occur (unlike Notebook 01's all-correct result).

In [2]:
RATING_ITEMS = [
    ("What is 7*8?", "56"),
    ("What is 7*8?", "54"),
    ("What is the capital of Italy?", "Rome"),
    ("What is the capital of Italy?", "Milan"),
    ("Name two prime numbers less than 10.", "2 and 3"),
    ("Name two prime numbers less than 10.", "4 and 6"),
    ("What is the freezing point of water in Celsius?", "0 degrees Celsius"),
    ("What is the freezing point of water in Celsius?", "32 degrees Celsius"),
    ("Who was the first president of the United States?", "George Washington"),
    ("Who was the first president of the United States?", "Thomas Jefferson"),
    ("What is the powerhouse of the cell?", "The mitochondria"),
    ("What is the powerhouse of the cell?", "The nucleus"),
    ("List the three primary colors.", "Red, blue, and yellow"),
    ("List the three primary colors.", "Red and blue"),
    ("What year did the Titanic sink?", "1912"),
    ("What year did the Titanic sink?", "Sometime in the early 1900s"),
    ("What is the chemical formula for water?", "H2O"),
    ("What is the chemical formula for water?", "Water is made of hydrogen and oxygen atoms"),
    ("Name the largest ocean on Earth.", "The Pacific Ocean"),
    ("Name the largest ocean on Earth.", "The Atlantic Ocean, which is the biggest"),
]
RUBRIC = "PASS if the response fully and correctly answers the question. FAIL if it is incorrect, incomplete, or does not address the question."
print(f"Real rating set fixed: {len(RATING_ITEMS)} items.")

Real rating set fixed: 20 items.


## 2. Two Real, Independently-Prompted LLM Raters

`[REAL]` Rater 1 and Rater 2 apply the identical real rubric, but through differently-worded, independently-written prompts -- two real, separate live API calls per item, not one call reused.

In [3]:
def rater_1(question, answer):
    prompt = (
        f"Rubric: {RUBRIC}\n\nQuestion: {question}\nAnswer: {answer}\n\n"
        "Apply the rubric. Reply with ONLY the single word PASS or FAIL."
    )
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, max_tokens=3,
    )
    text = resp.choices[0].message.content.strip().upper()
    return "PASS" if "PASS" in text else "FAIL"

def rater_2(question, answer):
    prompt = (
        f"You are grading a student's answer against this real standard: {RUBRIC}\n\n"
        f"Student was asked: {question}\nStudent answered: {answer}\n\n"
        "Grade the answer against the standard above. Respond with exactly one word: PASS or FAIL."
    )
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, max_tokens=3,
    )
    text = resp.choices[0].message.content.strip().upper()
    return "PASS" if "PASS" in text else "FAIL"

ratings = []
for question, answer in RATING_ITEMS:
    r1 = rater_1(question, answer)
    r2 = rater_2(question, answer)
    ratings.append({"question": question, "answer": answer, "rater_1": r1, "rater_2": r2, "agree": r1 == r2})
    print(f"Q: {question!r} A: {answer!r} -> rater_1={r1}, rater_2={r2}, agree={r1==r2}")

n_agree = sum(1 for r in ratings if r["agree"])
print(f"\nReal raw agreement: {n_agree}/{len(ratings)} = {n_agree/len(ratings)*100:.1f}%")
print("\n(pending real kappa computation)")

Q: 'What is 7*8?' A: '56' -> rater_1=PASS, rater_2=PASS, agree=True


Q: 'What is 7*8?' A: '54' -> rater_1=FAIL, rater_2=FAIL, agree=True


Q: 'What is the capital of Italy?' A: 'Rome' -> rater_1=PASS, rater_2=PASS, agree=True


Q: 'What is the capital of Italy?' A: 'Milan' -> rater_1=FAIL, rater_2=FAIL, agree=True


Q: 'Name two prime numbers less than 10.' A: '2 and 3' -> rater_1=PASS, rater_2=PASS, agree=True


...

Q: 'What year did the Titanic sink?' A: 'Sometime in the early 1900s' -> rater_1=FAIL, rater_2=FAIL, agree=True


Q: 'What is the chemical formula for water?' A: 'H2O' -> rater_1=PASS, rater_2=PASS, agree=True


Q: 'What is the chemical formula for water?' A: 'Water is made of hydrogen and oxygen atoms' -> rater_1=FAIL, rater_2=FAIL, agree=True


Q: 'Name the largest ocean on Earth.' A: 'The Pacific Ocean' -> rater_1=PASS, rater_2=PASS, agree=True


Q: 'Name the largest ocean on Earth.' A: 'The Atlantic Ocean, which is the biggest' -> rater_1=FAIL, rater_2=FAIL, agree=True

Real raw agreement: 20/20 = 100.0%

(pending real kappa computation)


**Real result:** `20/20 = 100.0%` raw agreement — the two real raters agreed on every single real item, including the three items deliberately written to be ambiguous/partial (`"Red and blue"`, `"Sometime in the early 1900s"`, `"Water is made of hydrogen and oxygen atoms"`) and the one confidently-wrong item (`"The Atlantic Ocean, which is the biggest"`). Both real raters independently marked every one of these `FAIL` — the rubric's `"fully and correctly"` / `"incomplete"` wording turned out to be strict and unambiguous enough, applied by this same underlying model twice, that no real disagreement occurred anywhere in this set.

## 3. Real Cohen's Kappa on the Real 2-Rater Confusion Table

`[COMPUTED FROM REAL DATA]` Building the real confusion table from Section 2's real ratings and computing Cohen's kappa directly, per Module 04's own formula.

In [4]:
pass_pass = sum(1 for r in ratings if r["rater_1"] == "PASS" and r["rater_2"] == "PASS")
pass_fail = sum(1 for r in ratings if r["rater_1"] == "PASS" and r["rater_2"] == "FAIL")
fail_pass = sum(1 for r in ratings if r["rater_1"] == "FAIL" and r["rater_2"] == "PASS")
fail_fail = sum(1 for r in ratings if r["rater_1"] == "FAIL" and r["rater_2"] == "FAIL")
total = pass_pass + pass_fail + fail_pass + fail_fail
print(f"Real confusion table: PASS-PASS={pass_pass}, PASS-FAIL={pass_fail}, FAIL-PASS={fail_pass}, FAIL-FAIL={fail_fail}, total={total}")

p_o = (pass_pass + fail_fail) / total
r1_pass = (pass_pass + pass_fail) / total
r1_fail = (fail_pass + fail_fail) / total
r2_pass = (pass_pass + fail_pass) / total
r2_fail = (pass_fail + fail_fail) / total
p_e = r1_pass * r2_pass + r1_fail * r2_fail
kappa = (p_o - p_e) / (1 - p_e) if p_e != 1 else float("nan")

print(f"Real p_o (observed agreement): {p_o:.4f}")
print(f"Real p_e (expected chance agreement): {p_e:.4f}")
print(f"Real Cohen's kappa: {kappa:.4f}")
print("\n(pending real interpretation)")

Real confusion table: PASS-PASS=10, PASS-FAIL=0, FAIL-PASS=0, FAIL-FAIL=10, total=20
Real p_o (observed agreement): 1.0000
Real p_e (expected chance agreement): 0.5000
Real Cohen's kappa: 1.0000

(pending real interpretation)


## 4. Real Interpretation

`[REAL]` A real, perfect Cohen's kappa: `p_o = 1.0000`, `p_e = 0.5000` (a real, balanced 10/10 PASS/FAIL split, so chance agreement itself was moderate, not trivially high), giving `κ = 1.0000` — full, honest agreement, not merely a raw-percentage artifact of an imbalanced label distribution the way Module 04's own hand-worked example demonstrated in the opposite direction (75% raw agreement, only κ=0.375). Reported exactly as it came out, without adjustment: **this result does not mean the rubric is trivial or that real rater agreement is guaranteed in general** — it means this specific real rubric wording, applied twice by the same underlying model via two differently-worded prompts, resolved every one of these 20 real items identically, including the ones deliberately designed to test edge cases. This is an honest, real, important limitation to state plainly: since both raters share the same underlying model, this result measures real *prompt-wording robustness* for a strict rubric, not genuine inter-rater diversity the way two structurally different models — or real human raters — might produce. A real test with a looser rubric, more genuinely ambiguous items, or two structurally different judge models would be a natural real next step to actually stress-test agreement rather than confirm robustness.